# 🛡️ GRACE: Automated Software Vulnerability Detection via In-Context Learning & Graph Similarity

**Sổ Tay Tái Tạo Kỹ Thuật Cho Môi Trường GPU Kaggle (v1.0.0)**

Sổ tay này triển khai tự động quy trình thử nghiệm phát hiện lỗ hổng phần mềm trên 2 tập dữ liệu benchmark thực tế: **Devign** (`DetectVul/devign`) và **Reveal** (`SensorLLM/Reveal`) bằng việc kết hợp:
1. **Stage 1**: Tự động chuẩn hóa dữ liệu & bóc tách mẫu cân bằng nhãn (**Stratified Slicing**).
2. **Stage 2**: Trích xuất đặc trưng với **CodeT5**, tính ma trận khoảng cách $L_2$, & lựa chọn mẫu ví dụ tối ưu theo công thức **Hybrid Reranking** ($0.7 \times \text{Jaccard} + 0.3 \times \text{GraphSim}$).
3. **Stage 3**: Lắp ráp câu lệnh dẫn hướng theo **Figure 6**, bóc tách nhãn với Parser Regex 4 tầng & bảo vệ tiến trình tuyệt đối bằng **JSONL Realtime Checkpointing**.
4. **Stage 4 & 5**: Đo lường trọn vẹn chỉ số **F1-Score, Precision, Recall** và xuất báo cáo nghiệm thu tự động.

---

## Bước 0: Thiết Lập Thư Mục Làm Việc Từ Dataset Đầu Vào (Dataset Setup)
Sao chép bộ mã nguồn từ thư mục Read-Only (`/kaggle/input/datasets/huuhieu3333/grace-source-code` hoặc `/kaggle/input/grace-source-code`) sang thư mục có quyền Ghi (`/kaggle/working/GRACE`) và chuyển thư mục làm việc.

In [ ]:
import os
import shutil

# 1. Tự động quét và tìm chính xác thư mục chứa run_pipeline.py
found_src_dir = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'run_pipeline.py' in files:
        found_src_dir = root
        break

dest_dir = '/kaggle/working/GRACE'
if os.path.exists(dest_dir):
    shutil.rmtree(dest_dir)

if found_src_dir:
    print(f'[*] Tìm thấy mã nguồn tại: {found_src_dir}')
    shutil.copytree(found_src_dir, dest_dir)
    print(f'✓ [OK] Đã sao chép toàn bộ mã nguồn sang: {dest_dir}')
else:
    print('[!] ERROR: Không tìm thấy file run_pipeline.py trong /kaggle/input!')

# 2. Chuyển thư mục làm việc về /kaggle/working/GRACE
%cd /kaggle/working/GRACE
!ls -la

## Bước 1: Chuẩn Bị Môi Trường & Thư Viện (Environment Setup)
Cài đặt các gói phụ thuộc cơ bản theo chuẩn `requirements.txt`.

In [ ]:
!pip install -q -r requirements.txt
print('✓ [Setup Complete] Các thư viện phụ thuộc đã sẵn sàng trên Kaggle!')

### Xác Minh Kết Nối Kaggle Secrets
Kiểm tra xem notebook đã được cấp quyền đọc `FPT_API_KEY` và `FPT_BASE_URL` từ Kaggle Secrets chưa.

In [ ]:
from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    api_key = user_secrets.get_secret('FPT_API_KEY')
    base_url = user_secrets.get_secret('FPT_BASE_URL')
    print(f'✓ [OK] Đã nạp thành công FPT_API_KEY ({api_key[:6]}...) và FPT_BASE_URL ({base_url})!')
except Exception as e:
    print(f'[!] Cảnh báo nạp Secret: {e}. Vui lòng kiểm tra Add-ons -> Secrets trên Kaggle.')

## Bước 2: Khởi Chạy Thực Nghiệm Toàn Diện Trên Bộ Dữ Liệu DEVIGN (100% Full Benchmark)
Trích xuất đặc trưng CodeT5 (768 chiều) cho 21,854 mẫu train làm Index và thực hiện suy luận bảo mật toàn diện trên toàn bộ **2,732 mẫu kiểm thử** với **FPT AI Factory API** (DeepSeek-V4-Flash / Gemma-4-26B).

*(Nhờ cơ chế Checkpointing JSONL Realtime, nếu phiên Kaggle bị timeout hoặc gián đoạn mạng, bạn chỉ cần bấm chạy lại cell này, hệ thống sẽ tự động bỏ qua các mẫu đã hoàn thành và tiếp tục ngay lập tức!)*

In [ ]:
!python run_pipeline.py --dataset DetectVul/devign --sample_ratio 1.0 --experiment_name devign_full_100pct

## Bước 3: Khởi Chạy Thực Nghiệm Mở Rộng Trên Bộ Dữ Liệu REVEAL (100% Full Benchmark)
Thử nghiệm trên toàn bộ dataset Reveal nhằm kiểm chứng khả năng tổng quát hóa (Generalizability) của phương pháp GRACE.

In [ ]:
!python run_pipeline.py --dataset SensorLLM/Reveal --sample_ratio 1.0 --experiment_name reveal_full_100pct

## Bước 5: Phỏng Đoán Báo Cáo Kết Quả Tự Động Xuất Xưởng (Artifact Inspection)
Trình diễn các file kết quả JSON và CSV đã được ghi xuống thư mục `output/`.

In [ ]:
import os
import pandas as pd
from pathlib import Path

output_dir = Path('/kaggle/working/output') if os.path.exists('/kaggle/working') else Path('./output')
print('Danh sách tệp nghiệm thu thu được từ hệ thống:')
if output_dir.exists():
    for f in output_dir.glob('*.*'):
        print('  ->', f.name)
else:
    print('Thư mục output chưa được tạo.')